# Concordância do GPT-4o com o novo dataset Gold

Este notebook compara as tags originais do GPT-4o com as tags do novo dataset gold. O join usa `example_id` no CSV original e `id` no CSV compartilhável.

> O GPT-4o participa da regra de maioria usada para construir o gold. Portanto, os números abaixo medem concordância com o consenso e tendem a ser otimistas. Eles não representam uma avaliação independente contra rótulos humanos.

## 1. Imports e configuração

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "analytics" / "src").is_dir()
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analytics.src.evaluate_mopep_tags import (
    ALLOWED_TAGS,
    compare_tags,
    macro_f1,
)

GPT4_ROOT = Path("/tmp/mopep-bmc-tags")
GOLDEN_ROOT = (
    REPO_ROOT / "analytics/results/mopep-golden-shareable"
)
OUTPUT_ROOT = (
    REPO_ROOT
    / "analytics/results/mopep-golden-evaluation/gpt4o"
)

SPLITS = ["train", "calibration", "test"]

## 2. Carregar e comparar cada split

In [ ]:
def compare_split(split):
    gpt4 = pd.read_csv(
        GPT4_ROOT / f"{split}.csv",
        usecols=["example_id", "tags"],
    ).rename(
        columns={"tags": "gpt4o_tags"}
    )

    golden = pd.read_csv(
        GOLDEN_ROOT / f"{split}.csv",
        usecols=["id", "tags"],
    ).rename(
        columns={"tags": "golden_tags"}
    )

    gpt4["gpt4o_tags"] = (
        gpt4["gpt4o_tags"].apply(json.loads)
    )
    golden["golden_tags"] = (
        golden["golden_tags"].apply(json.loads)
    )

    comparison = golden.merge(
        gpt4,
        left_on="id",
        right_on="example_id",
        how="inner",
    )

    scores = comparison.apply(
        lambda row: pd.Series(
            compare_tags(
                row["golden_tags"],
                row["gpt4o_tags"],
            )
        ),
        axis=1,
    )

    comparison = pd.concat(
        [comparison, scores],
        axis=1,
    )
    comparison["split"] = split

    return comparison.drop(columns="example_id")

In [ ]:
comparison_by_split = {
    split: compare_split(split)
    for split in SPLITS
}

comparison = pd.concat(
    comparison_by_split.values(),
    ignore_index=True,
)

comparison[
    [
        "id",
        "split",
        "golden_tags",
        "gpt4o_tags",
        "precision",
        "recall",
        "f1",
        "exact_match",
    ]
].head()

## 3. Resumo por split e geral

`mean_individual_f1` é a média do F1 calculado separadamente em cada exemplo. `macro_f1` calcula o F1 de cada tag e depois tira a média entre as tags.

In [ ]:
def summarize_comparison(frame, split):
    metric_input = frame.rename(
        columns={
            "golden_tags": "expected_tags",
            "gpt4o_tags": "generated_tags",
        }
    )

    return {
        "model": "gpt4o",
        "reference": "majority_vote_2_of_3",
        "split": split,
        "examples": len(frame),
        "mean_individual_precision": frame["precision"].mean(),
        "mean_individual_recall": frame["recall"].mean(),
        "mean_individual_f1": frame["f1"].mean(),
        "macro_f1": macro_f1(metric_input),
        "exact_match_rate": frame["exact_match"].mean(),
    }

In [ ]:
summary = pd.DataFrame(
    [
        summarize_comparison(
            comparison_by_split[split],
            split,
        )
        for split in SPLITS
    ]
    + [summarize_comparison(comparison, "all")]
)

summary

## 4. Métricas por tag

In [ ]:
def metrics_by_tag(frame, split):
    binarizer = MultiLabelBinarizer(
        classes=ALLOWED_TAGS
    )

    golden_matrix = binarizer.fit_transform(
        frame["golden_tags"]
    )
    gpt4_matrix = binarizer.transform(
        frame["gpt4o_tags"]
    )

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            golden_matrix,
            gpt4_matrix,
            average=None,
            zero_division=0,
        )
    )

    return pd.DataFrame(
        {
            "split": split,
            "tag": ALLOWED_TAGS,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "golden_support": support,
            "gpt4o_support": gpt4_matrix.sum(axis=0),
        }
    )

In [ ]:
per_tag = pd.concat(
    [
        metrics_by_tag(
            comparison_by_split[split],
            split,
        )
        for split in SPLITS
    ]
    + [metrics_by_tag(comparison, "all")],
    ignore_index=True,
)

per_tag.loc[
    per_tag["split"].eq("test")
].sort_values("f1")

## 5. Salvar os resultados

In [ ]:
OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

comparison_export = comparison.copy()

for column in [
    "golden_tags",
    "gpt4o_tags",
    "correct_tags",
    "missing_tags",
    "extra_tags",
]:
    comparison_export[column] = (
        comparison_export[column].apply(
            lambda values: json.dumps(
                values,
                ensure_ascii=False,
            )
        )
    )

comparison_path = OUTPUT_ROOT / "comparison.csv"
summary_path = OUTPUT_ROOT / "summary.csv"
per_tag_path = OUTPUT_ROOT / "per_tag.csv"

comparison_export.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8",
)
summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8",
)
per_tag.to_csv(
    per_tag_path,
    index=False,
    encoding="utf-8",
)

pd.DataFrame(
    {
        "artifact": [
            "comparison_by_example",
            "summary",
            "metrics_by_tag",
        ],
        "path": [
            comparison_path,
            summary_path,
            per_tag_path,
        ],
    }
)

## Interpretação

Para uma leitura principal, use a linha `test` de `summary`. Consulte `per_tag` para encontrar tags com menor concordância. Use `comparison` para inspecionar quais tags o GPT-4o deixou de incluir ou adicionou em relação ao consenso.